## ML Model for QRS metrics turned to Principal Components

The CSV file created using Stata

In [1]:
import joblib

import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor as RFR
from sklearn.linear_model import LinearRegression as LR
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

In [2]:
model_param_file = "../model/rfr_pc_params.json"
rf_model_file = "../model/rfr_pc_best_estimator.joblib"

In [3]:
DATA_DIR = "../data"
df = pd.read_csv(f"{DATA_DIR}/principal_factors/qrs_amplitudes_principalfactors.csv")
df_time = pd.read_csv(f"{DATA_DIR}/qrs_metrics_subject.csv")
df = df.merge(
    df_time[['file_num', 'subject_idx', 'median_dur']],
    on=['file_num', 'subject_idx']
)

df.to_csv(f"{DATA_DIR}/principal_factors/principle_factors_duration_metadata.csv", index=False)

In [4]:
model_columns = [col for col in df.columns if "pf" in col]
model_columns += [
    "is_male",
    "normal_ecg",
    "median_dur",
]
df.head()

,file_num,subject_idx,exam_id,age,nn_predicted_age,is_male,normal_ecg,pf1_overall_amplitude,pf2_limb_minus_chest,pf3_rt_sternum_axis,pf4_foot_axis,pf5_lt_armpit_axis,median_dur
0,1,0,4409337,70,94.596581,False,False,0.135067,-0.504476,-0.005761,-0.276552,-0.468117,33.0
1,1,1,835969,63,61.776833,True,True,0.545185,0.108943,1.697019,0.253758,-1.710396,59.0
2,1,2,2668899,47,46.891685,False,False,0.550287,0.117389,-0.642631,-0.760782,-0.387992,35.0
3,1,3,884099,37,69.076965,False,False,1.152258,1.447604,0.153328,-0.265055,0.206747,35.5
4,1,4,526403,43,41.630150,False,True,-0.249599,0.146396,0.114243,0.656242,0.603728,38.5


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    df[model_columns],
    df['age'],  # Replace 'target_column' with the actual name of your target column
    test_size=0.2,
    random_state=42
)

rfr = RFR(
    max_depth=5,
    min_samples_split=5,
    n_jobs=-1,
    random_state=314,
)

rfr.fit(X_train[model_columns], y_train)
y_train_pred = rfr.predict(X_train[model_columns])
y_test_pred = rfr.predict(X_test[model_columns])

print(f"Training R2: {round(r2_score(y_train, y_train_pred), 3)}")
print(f"Testing R2: {round(r2_score(y_test, y_test_pred), 3)}")

Training R2: 0.242
Testing R2: 0.237


In [6]:
lst = []
for col, feat_imp in zip(model_columns, rfr.feature_importances_):
    lst.append([col, round(feat_imp, 3)])

pd.DataFrame(lst, columns=['feature', 'importance']).sort_values('importance', ascending=False)

,feature,importance
4,pf5_lt_armpit_axis,0.498
6,normal_ecg,0.211
1,pf2_limb_minus_chest,0.139
2,pf3_rt_sternum_axis,0.059
3,pf4_foot_axis,0.049
7,median_dur,0.044
0,pf1_overall_amplitude,0.000
5,is_male,0.000


In [8]:
from sklearn.model_selection import GridSearchCV, KFold

parameters = {
    'n_estimators': [200],                  # more trees doesn't cause overfitting; just fix it
    'max_depth': [4, 6, 8, 10, 12],
    'min_samples_leaf': [20, 50, 100, 200], # sized for ~74k rows
    'max_features': [0.33, 0.5, 1.0],       # decorrelates trees
    'max_samples': [0.3, 0.6, None],        # bootstrap subsample size
}

cv = KFold(n_splits=5, shuffle=True, random_state=314)
clf = GridSearchCV(
    RFR(n_jobs=-1, random_state=314),
    parameters,
    cv=cv,
    scoring='r2',
    return_train_score=True,   # lets you see the train/val gap for every combo
    n_jobs=1,
)
clf.fit(X_train[model_columns], y_train)


,estimator,RandomForestR...dom_state=314)
,param_grid,"{'max_depth': [4, 6, ...], 'max_features': [0.33, 0.5, ...], 'max_samples': [0.3, 0.6, ...], 'min_samples_leaf': [20, 50, ...], ...}"
,scoring,'r2'
,n_jobs,1
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,True
,n_estimators,200


In [9]:
res = pd.DataFrame(clf.cv_results_)
res['gap'] = res['mean_train_score'] - res['mean_test_score']
res.sort_values('mean_test_score', ascending=False)[
    ['params', 'mean_train_score', 'mean_test_score', 'std_test_score', 'gap']
].head(15)

,params,mean_train_score,mean_test_score,std_test_score,gap
164,"{'max_depth': 12, 'max_features': 0.5, 'max_sa...",0.367131,0.281292,0.004571,0.085839
160,"{'max_depth': 12, 'max_features': 0.5, 'max_sa...",0.349648,0.280663,0.004788,0.068985
152,"{'max_depth': 12, 'max_features': 0.33, 'max_s...",0.344947,0.280257,0.003956,0.064690
148,"{'max_depth': 12, 'max_features': 0.33, 'max_s...",0.331570,0.279274,0.004007,0.052296
172,"{'max_depth': 12, 'max_features': 1.0, 'max_sa...",0.361592,0.279098,0.005082,0.082494
176,"{'max_depth': 12, 'max_features': 1.0, 'max_sa...",0.381922,0.278844,0.005112,0.103078
156,"{'max_depth': 12, 'max_features': 0.5, 'max_sa...",0.324956,0.278796,0.004440,0.046160
165,"{'max_depth': 12, 'max_features': 0.5, 'max_sa...",0.328345,0.278056,0.004691,0.050288
168,"{'max_depth': 12, 'max_features': 1.0, 'max_sa...",0.333010,0.277968,0.004858,0.055041
128,"{'max_depth': 10, 'max_features': 0.5, 'max_sa...",0.333704,0.277908,0.004480,0.055796


In [10]:
y_train_pred = clf.predict(X_train[model_columns])
y_test_pred = clf.predict(X_test[model_columns])

print(f"Training R2: {round(r2_score(y_train, y_train_pred), 3)}")
print(f"Testing R2: {round(r2_score(y_test, y_test_pred), 3)}")

Training R2: 0.362
Testing R2: 0.281


In [12]:
import numpy as np
np.sqrt(0.281)

np.float64(0.5300943312279429)

In [13]:
dir(clf)

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__sklearn_clone__',
 '__sklearn_tags__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_build_request_for_signature',
 '_check_refit_for_multimetric',
 '_check_scorers_accept_sample_weight',
 '_doc_link_module',
 '_doc_link_template',
 '_doc_link_url_param_generator',
 '_estimator_type',
 '_format_results',
 '_get_default_requests',
 '_get_doc_link',
 '_get_metadata_request',
 '_get_param_names',
 '_get_params_html',
 '_get_routed_params_for_fit',
 '_get_scorers',
 '_html_repr',
 '_parameter_constraints',
 '_repr_html_',
 '_repr

In [16]:
clf.estimator.fit(X_train[model_columns], y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [17]:
lst = []
for col, importances in zip(model_columns, clf.estimator.feature_importances_):
    lst.append([col, round(importances, 3)])

pd.DataFrame(lst, columns=['feature', 'importance']).sort_values('importance', ascending=False)

,feature,importance
4,pf5_lt_armpit_axis,0.245
1,pf2_limb_minus_chest,0.163
2,pf3_rt_sternum_axis,0.149
3,pf4_foot_axis,0.145
0,pf1_overall_amplitude,0.129
7,median_dur,0.104
6,normal_ecg,0.049
5,is_male,0.016


In [7]:
# Grid search code

parameters = {
    'n_estimators': [25, 50, 75, 100],
    'min_samples_split': [3, 5],
    'min_samples_leaf': [3, 5, 7],
    # 'bootstrap': [True, False],
    'oob_score': [True],
    'min_impurity_decrease': [0.005, 0.0005],
    'n_jobs': [-1],
    'random_state': [314],
}
model = RFR()
clf = GridSearchCV(model, parameters)

# if not Path(rf_model_file).exists():
if True:
    clf.fit(X_train[model_columns], y_train)
    joblib.dump(clf.best_estimator_, rf_model_file)
else:
    clf.best_estimator_ = joblib.load(rf_model_file)

In [8]:
clf.best_estimator_

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,3
,min_samples_leaf,7
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.005
,bootstrap,True
,oob_score,True


In [9]:
y_train_pred = clf.predict(X_train[model_columns])
y_test_pred = clf.predict(X_test[model_columns])

print(f"Training R2: {round(r2_score(y_train, y_train_pred), 3)}")
print(f"Testing R2: {round(r2_score(y_test, y_test_pred), 3)}")

Training R2: 0.603
Testing R2: 0.273


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    df[model_columns],
    df['nn_predicted_age'],  # Replace 'target_column' with the actual name of your target column
    test_size=0.2,
    random_state=42
)

rfr = RFR(
    max_depth=5,
    min_samples_split=5,
    n_jobs=-1,
    random_state=314,
)

rfr.fit(X_train[model_columns], y_train)
y_train_pred = rfr.predict(X_train[model_columns])
y_test_pred = rfr.predict(X_test[model_columns])

print(f"Training R2: {round(r2_score(y_train, y_train_pred), 3)}")
print(f"Testing R2: {round(r2_score(y_test, y_test_pred), 3)}")

Training R2: 0.327
Testing R2: 0.321
